# 🔍 Notebook 01: Exploración Inicial y Auditoría de Calidad de Datos Clínicos
## Proyecto de Predicción de Riesgo Cardiovascular — Cohorte Framingham
**Autora:** Ana Colina Arismendi  
**Fuente:** *Framingham Heart Study (NIH / National Heart, Lung, and Blood Institute)*

> **Objetivo:** Realizar una auditoría técnica y médica exhaustiva del dataset bruto `framingham.csv` (4,240 registros longitudinales), identificando estructura, cardinalidad, valores nulos, duplicados y coherencia fisiológica de los biomarcadores continuos (colesterol, glucemia, presión arterial, frecuencia cardíaca).

---

### 📑 Contenido del Notebook
1. [Configuración del entorno y librerías](#1)
2. [Carga e inspección preliminar de datos](#2)
3. [Estructura, tipos de datos y memoria](#3)
4. [Estadísticos descriptivos globales](#4)
5. [Auditoría de valores nulos y completitud](#5)
6. [Auditoría de duplicados técnicos](#6)
7. [Detección de anomalías hemodinámicas y fisiológicas](#7)
8. [Distribución univariada de factores de riesgo](#8)
9. [Inspección de la variable objetivo (TenYearCHD / cardio)](#9)
10. [Conclusiones del diagnóstico y hoja de ruta](#10)


<a id="1"></a>
## 1. Configuración del entorno y librerías

Importamos las bibliotecas fundamentales para análisis exploratorio, cálculo matricial y visualización estadística avanzada.


In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_theme(style='whitegrid', palette='mako')

print(f"✓ Pandas: {pd.__version__} | NumPy: {np.__version__}")


✓ Pandas: 2.3.3 | NumPy: 2.3.5


<a id="2"></a>
## 2. Carga e inspección preliminar de datos

Cargamos el dataset original `framingham.csv`. A diferencia de estudios transversales con sesgo de supervivencia, este estudio prospectivo siguió a cada participante durante una década para registrar la aparición de cardiopatía coronaria.


In [2]:
ruta_raw = '../Data/framingham.csv'
if not os.path.exists(ruta_raw):
    ruta_raw = 'Data/framingham.csv'

df_raw = pd.read_csv(ruta_raw)
print(f"Dimensiones de la cohorte: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} variables")
df_raw.head(10)


Dimensiones de la cohorte: 4,240 filas × 16 variables


<a id="3"></a>
## 3. Estructura, tipos de datos y memoria

Verificamos los tipos de datos asignados y la cardinalidad de cada columna clínica.


In [3]:
print("--- Resumen Estructural de la Cohorte ---")
df_raw.info()

print("\n--- Cardinalidad de Variables ---")
cardinalidad = pd.DataFrame({
    'Tipo': df_raw.dtypes,
    'Valores_Unicos': df_raw.nunique(),
    'Pct_Unicos': (df_raw.nunique() / len(df_raw) * 100).round(2),
    'Ejemplo_Valores': [df_raw[c].dropna().unique()[:4].tolist() for c in df_raw.columns]
})
print(cardinalidad)


--- Resumen Estructural de la Cohorte ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4240 entries, 0 to 4239
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   male             4240 non-null   int64  
 1   age              4240 non-null   int64  
 2   education        4135 non-null   float64
 3   currentSmoker    4240 non-null   int64  
 4   cigsPerDay       4211 non-null   float64
 5   BPMeds           4187 non-null   float64
 6   prevalentStroke  4240 non-null   int64  
 7   prevalentHyp     4240 non-null   int64  
 8   diabetes         4240 non-null   int64  
 9   totChol          4190 non-null   float64
 10  sysBP            4240 non-null   float64
 11  diaBP            4240 non-null   float64
 12  BMI              4221 non-null   float64
 13  heartRate        4239 non-null   float64
 14  glucose          3852 non-null   float64
 15  TenYearCHD       4240 non-null   int64  
dtypes: float64(9), int

<a id="4"></a>
## 4. Estadísticos descriptivos globales

Examinamos las medidas de tendencia central, dispersión, asimetría y valores extremos de las variables continuas.


In [4]:
num_cols = ['age', 'cigsPerDay', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose']
desc = df_raw[num_cols].describe().T
desc['IQR'] = desc['75%'] - desc['25%']
desc['Skewness'] = df_raw[num_cols].skew()
desc['Kurtosis'] = df_raw[num_cols].kurtosis()
print("--- Estadísticos Descriptivos Robustos ---")
print(desc.round(2))


--- Estadísticos Descriptivos Robustos ---
             count    mean    std     min     25%    50%     75%    max    IQR  Skewness  Kurtosis
age         4240.0   49.58   8.57   32.00   42.00   49.0   56.00   70.0  14.00      0.23     -0.99
cigsPerDay  4211.0    9.01  11.92    0.00    0.00    0.0   20.00   70.0  20.00      1.25      1.02
totChol     4190.0  236.70  44.59  107.00  206.00  234.0  263.00  696.0  57.00      0.87      4.13
sysBP       4240.0  132.35  22.03   83.50  117.00  128.0  144.00  295.0  27.00      1.15      2.16
diaBP       4240.0   82.90  11.91   48.00   75.00   82.0   90.00  142.5  15.00      0.71      1.28
BMI         4221.0   25.80   4.08   15.54   23.07   25.4   28.04   56.8   4.97      0.98      2.66
heartRate   4239.0   75.88  12.03   44.00   68.00   75.0   83.00  143.0  15.00      0.64      0.91
glucose     3852.0   81.96  23.95   40.00   71.00   78.0   87.00  394.0  16.00      6.21     58.70


<a id="5"></a>
## 5. Auditoría de valores nulos y completitud

Identificamos columnas con valores faltantes que requerirán imputación médica razonada en el preprocesamiento.


In [5]:
nulos = pd.DataFrame({
    'Total_Nulos': df_raw.isnull().sum(),
    'Porcentaje_Nulos': (df_raw.isnull().mean() * 100).round(2)
})
nulos_con_datos = nulos[nulos['Total_Nulos'] > 0].sort_values(by='Total_Nulos', ascending=False)
print("--- Variables con Valores Faltantes ---")
print(nulos_con_datos)

fig, ax = plt.subplots(figsize=(8, 4))
nulos_con_datos['Porcentaje_Nulos'].plot(kind='bar', ax=ax, color='#e74c3c')
ax.set_ylabel('% Faltante')
ax.set_title('Porcentaje de Valores Nulos por Variable Clínica', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


--- Variables con Valores Faltantes ---
            Total_Nulos  Porcentaje_Nulos
glucose             388              9.15
education           105              2.48
BPMeds               53              1.25
totChol              50              1.18
cigsPerDay           29              0.68
BMI                  19              0.45
heartRate             1              0.02


<a id="6"></a>
## 6. Auditoría de duplicados técnicos

Verificamos la unicidad de las observaciones registradas.


In [6]:
dup_exactos = df_raw.duplicated().sum()
print(f"Duplicados exactos en la cohorte: {dup_exactos}")


Duplicados exactos en la cohorte: 0


<a id="7"></a>
## 7. Detección de anomalías hemodinámicas y límites fisiológicos

Comprobamos si existen incongruencias biológicas fundamentales, como presión diastólica superior a sistólica o valores de glucemia incompatibles con la vida.


In [7]:
print("--- Auditoría de Plausibilidad Fisiológica ---")
anom_sys_dia = (df_raw['sysBP'] <= df_raw['diaBP']).sum()
print(f"Casos con Presión Diastólica >= Sistólica: {anom_sys_dia}")

anom_pp_baja = ((df_raw['sysBP'] - df_raw['diaBP']) < 15).sum()
print(f"Casos con Presión de Pulso patológicamente baja (<15 mmHg): {anom_pp_baja}")

anom_chol_extremo = ((df_raw['totChol'] < 90) | (df_raw['totChol'] > 600)).sum()
print(f"Casos con Colesterol fuera de rango biológico habitual (<90 o >600 mg/dL): {anom_chol_extremo}")

anom_bmi_extremo = ((df_raw['BMI'] < 14) | (df_raw['BMI'] > 60)).sum()
print(f"Casos con IMC extremo (<14 o >60): {anom_bmi_extremo}")


--- Auditoría de Plausibilidad Fisiológica ---
Casos con Presión Diastólica >= Sistólica: 0
Casos con Presión de Pulso patológicamente baja (<15 mmHg): 0
Casos con Colesterol fuera de rango biológico habitual (<90 o >600 mg/dL): 1
Casos con IMC extremo (<14 o >60): 0


<a id="8"></a>
## 8. Distribución univariada de factores de riesgo

Visualizamos las distribuciones de los principales biomarcadores para evaluar simetría y presencia de colas pesadas.


In [8]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
vars_grafico = ['age', 'sysBP', 'diaBP', 'totChol', 'glucose', 'cigsPerDay']
titulos = ['Edad (años)', 'Presión Sistólica (mmHg)', 'Presión Diastólica (mmHg)',
           'Colesterol Total (mg/dL)', 'Glucosa en Ayuno (mg/dL)', 'Cigarrillos al Día']

for ax, col, tit in zip(axes.flatten(), vars_grafico, titulos):
    sns.histplot(df_raw[col].dropna(), kde=True, ax=ax, color='#16a085', bins=30)
    ax.set_title(tit, fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()


<a id="9"></a>
## 9. Inspección de la variable objetivo (`TenYearCHD`)

La variable objetivo binaria indica si el paciente presentó un evento coronario documentado en los 10 años posteriores al examen basal.


In [9]:
conteo_target = df_raw['TenYearCHD'].value_counts()
pct_target = df_raw['TenYearCHD'].value_counts(normalize=True) * 100

print("--- Distribución de Incidencia a 10 Años (TenYearCHD) ---")
for val, count in conteo_target.items():
    lbl = "Libre de Evento Coronario (0)" if val == 0 else "Con Evento Coronario a 10a (1)"
    print(f"  {lbl}: {count:,} pacientes ({pct_target[val]:.2f}%)")

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Sin Evento (0)', 'Evento a 10a (1)'], conteo_target.values, color=['#2ecc71', '#e74c3c'], width=0.5)
for bar, c, p in zip(bars, conteo_target.values, pct_target.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, f"{c:,}\n({p:.1f}%)", ha='center', fontweight='bold')
ax.set_ylabel('Pacientes')
ax.set_title('Incidencia Acumulada de Cardiopatía Coronaria a 10 Años', fontsize=12)
ax.set_ylim(0, conteo_target.max() * 1.2)
plt.tight_layout()
plt.show()


--- Distribución de Incidencia a 10 Años (TenYearCHD) ---
  Libre de Evento Coronario (0): 3,596 pacientes (84.81%)
  Con Evento Coronario a 10a (1): 644 pacientes (15.19%)


<a id="10"></a>
## 10. Conclusiones del diagnóstico y plan de preprocesamiento

### Resumen del Hallazgo Clínico
1. **Calidad Fisiológica:** El dataset Framingham exhibe coherencia hemodinámica estricta ($sysBP > diaBP$ en el 100% de los casos).
2. **Imputación Necesaria:** Variables como `glucose` (9.2% nulos), `totChol` (1.2% nulos), `BMI` (0.4% nulos) y `cigsPerDay` (0.7% nulos) requieren imputación fundamentada (por ejemplo, mediana según estatus de tabaquismo o diabetes).
3. **Desbalance Epidemiológico:** La incidencia es del ~15%, representativa de una cohorte comunitaria real. No debe aplicarse oversampling agresivo sin justificación, sino ponderación de pérdida (`class_weight='balanced'`) y evaluación por ROC-AUC y F1.
4. **Próximo Paso:** Proceder al **Notebook 02** aplicando la guía clínica de 19 pasos para estandarizar, imputar y generar variables derivadas hemodinámicas.
